In [1]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Base directory for AntM2C_10M
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"

# Structured image directory (validated in 01_image_path_validation)
image_dir = f"{base_dir}/AntM2C_image_structured"

# Path to CSV containing item_id -> structured image path
image_paths_csv = f"{base_dir}/AntM2C_preprocessed/image_paths_structured.csv"

print("Image directory:", image_dir)
print("CSV path:", image_paths_csv)

Image directory: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_image_structured
CSV path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/image_paths_structured.csv


In [16]:
import os
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, AutoModel

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Loading Pretrained ViT Model
Model: google/vit-base-patch16-224-in21k

*   Output embedding size = 768
*   Uses AutoImageProcessor for resizing / normalization
*   Sets eval() and no_grad() for efficient inference

In [8]:
# Pretrained model name
vit_model_name = "google/vit-base-patch16-224-in21k"

# Load processor (handles resize, crop, normalization)
processor = AutoImageProcessor.from_pretrained(vit_model_name)

# Load pretrained ViT (encoder only, no classification head needed)
model = AutoModel.from_pretrained(vit_model_name)
model = model.to(device)
model.eval()

print("Loaded ViT model:", vit_model_name)

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loaded ViT model: google/vit-base-patch16-224-in21k


Passing a dummy image to check the output

In [ ]:
# Create a dummy batch of 1 image (random)
dummy_image = torch.rand(1, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = model(pixel_values=dummy_image)
    cls_embedding = outputs.last_hidden_state[:, 0, :]

print("CLS embedding shape:", cls_embedding.shape)

CLS embedding shape: torch.Size([1, 768])


This confirms that the model is loaded correctly, and that it is configured, and the output dimensions for image will be 768-dim vectors

## Load and validate Image Metadata

In this step, we load the image_paths_structured.csv file generated during preprocessing.

*   item_id -> unique AntM2C item identifier
*   image_path -> structured folder path (e.g., 287/28721.png)

The goal is to ensure that the Vision Transformer receives only valid, aligned image paths so that the embedding matrix indexing matches the item_id list perfectly.

In [9]:
# Load metadata CSV
df = pd.read_csv(image_paths_csv)

print("Total rows in CSV:", len(df))
df.head()

Total rows in CSV: 59142


,item_id,image_path
0,28721,287/28721.png
1,287697,287/287697.png
2,287443,287/287443.png
3,28734,287/28734.png
4,287655,287/287655.png


In [14]:
# Clean and normalize image paths
df["image_path"] = (
    df["image_path"]
    .astype(str)
    .str.strip()
    .str.lstrip("/\\")
)

valid_items = []
missing_count = 0
debug_shown = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    item_id = int(row["item_id"])
    img_path = row["image_path"]

    full_path = os.path.join(image_dir, img_path)

    if not os.path.exists(full_path):
        missing_count += 1
        if debug_shown < 3:
            print("[Missing file]", full_path)
            debug_shown += 1
        continue

    valid_items.append((item_id, full_path))

print(f"Valid images found: {len(valid_items)}")
print(f"Missing images skipped: {missing_count}")

100%|██████████| 59142/59142 [02:22<00:00, 416.19it/s] 

Valid images found: 59142
Missing images skipped: 0


## Fast Image Preprocessing + Streaming Batching


In this step, we prepare all images for the Vision Transformer (ViT). Although ViT uses its own preprocessing pipeline via AutoImageProcessor, we still need a reliable method to load images ffrom the disk, apply preprocessing, batch images, skip corrupted images and prepare batched for feature extractions.

This optimized version uses:
*   PyTorch Dataset for efficient loading
*   ImageProcessor for on-the-fly ViT preprocessing
*   DataLoader with 4 workers for parallel I/O
*   Streaming batches directly into ViT

In [17]:
class AntM2CImageDataset(Dataset):
    def __init__(self, items, processor):
        self.items = items
        self.processor = processor

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item_id, img_path = self.items[idx]

        try:
            img = Image.open(img_path).convert("RGB")
            pixel = self.processor(img, return_tensors="pt")["pixel_values"][0]
            return item_id, pixel
        except:
            return None  # skip corrupted images

Data Loader (Parellel loading)

In [18]:
dataset = AntM2CImageDataset(valid_items, processor)

loader = DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    shuffle=False,
    collate_fn=lambda x: [i for i in x if i is not None]
)

print("Total images:", len(dataset))

Total images: 59142


In [19]:
all_embeddings = []
all_item_ids = []

model.eval()

with torch.no_grad():
    for batch in tqdm(loader):
        ids = []
        pixels = []

        for item_id, pixel in batch:
            ids.append(item_id)
            pixels.append(pixel)

        pixels = torch.stack(pixels).to(device)  # Move to GPU

        # Forward pass through ViT
        outputs = model(pixel_values=pixels)
        cls_embeds = outputs.last_hidden_state[:, 0, :].cpu()  # (B, 768)

        all_embeddings.append(cls_embeds)
        all_item_ids.extend(ids)

# Concatenate everything
all_embeddings = torch.cat(all_embeddings, dim=0)

100%|██████████| 925/925 [1:57:58<00:00,  7.65s/it]


In [22]:
!git config --global user.email "sharmfernando33@gmail.com"
!git config --global user.name “sheebaboy33"

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file
